# JSON and JSONL Parsing: Preserve Record Boundaries

| Field | Value |
|---|---|
| Stage | Data foundation |
| Difficulty | Intermediate |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
Choose records with an explicit JSON path and stable IDs; do not flatten an entire nested payload into an untraceable string.

## 30-Second Summary

This notebook parses nested employee JSON and line-delimited event JSONL using the standard library. It emits one employee or event record per document and validates IDs, line provenance, and malformed-line isolation.

## Why This Matters

Nested arrays describe different grains, while JSONL is designed for independent records. Treating both as one blob makes updates, errors, and citations unnecessarily broad.

## Scope

| Covers | Does not cover |
|---|---|
| Nested paths, JSONL lines, record IDs, typed metadata | Streaming huge files, schema registries, arbitrary flattening, remote APIs |


## Mental Model

```text
JSON  -> select $.employees[*] -> employee documents
JSONL -> parse each line        -> event documents (line provenance)
```


In [1]:
from pathlib import Path
import json

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file(): return candidate
    raise FileNotFoundError("Run this notebook from inside the repository.")

REPO_ROOT = find_repo_root()
JSON_DIR = REPO_ROOT / "05-DataIngestParsing/data/json_files"
COMPANY_PATH, EVENTS_PATH = JSON_DIR / "company_data.json", JSON_DIR / "events.jsonl"
company_payload = json.loads(COMPANY_PATH.read_text(encoding="utf-8"))
len(company_payload["employees"]), company_payload["company"]


(2, 'TechCorp')

## How It Works

For nested JSON we select a named array and build IDs from source-owned keys. For JSONL we parse and validate one line at a time, so one malformed record can be reported without losing the rest of the file.


## Baseline

The baseline keeps each entire file as raw text. It preserves bytes but mixes employee, department, and event grains into broad retrieval units.


In [2]:
baseline_documents = [
    {"source": path.relative_to(REPO_ROOT).as_posix(), "content": path.read_text(encoding="utf-8")}
    for path in (COMPANY_PATH, EVENTS_PATH)
]
[(item["source"], len(item["content"])) for item in baseline_documents]


[('05-DataIngestParsing/data/json_files/company_data.json', 1037),
 ('05-DataIngestParsing/data/json_files/events.jsonl', 232)]

## Technique Implementation

Employee documents retain skills and project lists; event documents retain typed fields plus the one-based source line. Unknown event fields remain in metadata rather than being silently discarded.


In [3]:
company_source = COMPANY_PATH.relative_to(REPO_ROOT).as_posix()
employee_documents = []
for employee in company_payload["employees"]:
    projects = ", ".join(f"{item['name']} ({item['status']})" for item in employee["projects"])
    employee_documents.append({
        "id": f"employee:{employee['id']}", "source": company_source,
        "metadata": employee,
        "content": f"Name: {employee['name']} | Role: {employee['role']} | Skills: {', '.join(employee['skills'])} | Projects: {projects}",
    })

event_documents, malformed_lines = [], []
events_source = EVENTS_PATH.relative_to(REPO_ROOT).as_posix()
for line_number, line in enumerate(EVENTS_PATH.read_text(encoding="utf-8").splitlines(), start=1):
    try:
        event = json.loads(line)
        event_documents.append({
            "id": f"event:{line_number}", "source": events_source,
            "line": line_number, "metadata": event,
            "content": " | ".join(f"{key}: {value}" for key, value in event.items()),
        })
    except json.JSONDecodeError as error:
        malformed_lines.append({"line": line_number, "error": str(error)})

[(doc["id"], doc["content"]) for doc in employee_documents + event_documents]


[('employee:1',
  'Name: John Doe | Role: Software Engineer | Skills: Python, JavaScript, React | Projects: RAG System (In Progress), Data Pipeline (Completed)'),
 ('employee:2',
  'Name: Jane Smith | Role: Data Scientist | Skills: Python, Machine Learning, SQL | Projects: ML Model (In Progress), Analytics Dashboard (Planning)'),
 ('event:1', 'timestamp: 2024-01-01 | event: user_login | user_id: 123'),
 ('event:2',
  'timestamp: 2024-01-01 | event: page_view | user_id: 123 | page: /home'),
 ('event:3',
  'timestamp: 2024-01-01 | event: purchase | user_id: 123 | amount: 99.99')]

## Controlled Experiment

We compare raw-file context with the employee record answering a question about the Data Scientist's skills and projects. We also verify that every JSONL line produces exactly one event document and no hidden parse failures.


In [4]:
top_employee = next(doc for doc in employee_documents if doc["metadata"]["role"] == "Data Scientist")
experiment_result = {
    "employee_records": len(employee_documents),
    "event_records": len(event_documents),
    "malformed_lines": len(malformed_lines),
    "answer_name": top_employee["metadata"]["name"],
    "record_characters": len(top_employee["content"]),
    "raw_company_characters": len(baseline_documents[0]["content"]),
}
experiment_result


{'employee_records': 2,
 'event_records': 3,
 'malformed_lines': 0,
 'answer_name': 'Jane Smith',
 'record_characters': 146,
 'raw_company_characters': 1037}

## Evaluation

The nested payload yields **2 employee records** and the JSONL file yields **3 event records**, with no malformed lines. The Data Scientist record resolves to Jane Smith and is much smaller than the full company payload.


In [5]:
assert experiment_result == {
    "employee_records": 2, "event_records": 3, "malformed_lines": 0,
    "answer_name": "Jane Smith", "record_characters": experiment_result["record_characters"],
    "raw_company_characters": experiment_result["raw_company_characters"],
}
assert len({doc["id"] for doc in employee_documents + event_documents}) == 5
assert experiment_result["record_characters"] < experiment_result["raw_company_characters"]
print("JSON checks passed for nested JSON and JSONL.")


JSON checks passed for nested JSON and JSONL.


## Decision Guide

| Shape | Grain |
|---|---|
| Array of entities | One document per entity |
| JSONL event stream | One document per line/event |
| Nested configuration | Section by stable path |
| Frequently filtered fields | Keep typed metadata; filter before ranking |


## Failure Modes and Debugging

| Symptom | Cause | Fix |
|---|---|---|
| One bad line kills batch | Whole-file JSONL parse | Parse/isolate by line |
| IDs change | Array index used as identity | Use source keys plus version |
| Nested facts vanish | Over-aggressive flattening | Define and test selected paths |
| Numbers/dates become ambiguous | String-only serialization | Preserve typed metadata/schema |


## Production Notes

### Observability
Track records, malformed lines, missing keys, schema versions, unknown fields, and duplicates.

### Safety and Guardrails
Bound nesting depth and record size; redact sensitive fields before logs or embeddings.

### Latency and Cost
Stream large JSONL files and checkpoint progress rather than loading them entirely.


## Practice

Append one malformed JSONL line and one valid event with a new field. Verify isolation and decide whether schema drift is accepted or quarantined.

## Recall

Toggle - Recall: Why is JSON path selection important?
It defines the entity grain and prevents unrelated nested objects from becoming one blob.

Toggle - Recall: Why keep JSONL line numbers?
They locate failures and support source-level citations.

## Sources

- [Python `json` documentation](https://docs.python.org/3/library/json.html)
- [JSON Lines format](https://jsonlines.org/)
- Repository JSON fixtures

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for the documented fixture shapes | Add malformed, deeply nested, and schema-drift fixtures |
